# 量化投资 — 数据探索第一课

**目标**：用 Python 拉取真实的A股数据，掌握量化分析的基本功。

本 Notebook 涵盖：
1. 用 akshare 获取沪深300历史数据
2. 绘制K线图与移动平均线
3. 计算收益率、波动率、夏普比率
4. 计算最大回撤
5. 多只股票的相关性矩阵
6. 手动模拟一个双均线策略的收益曲线

---
## 核心概念速查

| 概念 | 公式 | 含义 |
|------|------|------|
| **简单收益率** | $R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$ | 单期价格变动百分比 |
| **对数收益率** | $r_t = \ln(\frac{P_t}{P_{t-1}})$ | 时间可加性，更符合正态假设 |
| **年化收益率** | $R_{annual} = (1 + \bar{R})^{252} - 1$ | 折算成年化，A股一年约252个交易日 |
| **年化波动率** | $\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$ | 风险度量，越大越不稳定 |
| **夏普比率** | $Sharpe = \frac{R_{annual} - R_f}{\sigma_{annual}}$ | 每单位风险的超额收益，>1不错，>2优秀 |
| **最大回撤** | $MDD = \min_t(\frac{P_t - \max_{s \leq t}P_s}{\max_{s \leq t}P_s})$ | 从峰值到谷底的最大亏损幅度 |
| **Alpha** | 策略收益 - 基准收益（去市场波动后）| 策略的超额收益能力 |
| **Beta** | $\beta = \frac{Cov(R_s, R_m)}{Var(R_m)}$ | 策略对市场的敏感度 |

## 1. 导入依赖 & 环境准备

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
import random
warnings.filterwarnings("ignore")

# --- 尝试导入 akshare ---
try:
    import akshare as ak
    HAS_AKSHARE = True
except ImportError:
    HAS_AKSHARE = False
    print("⚠ akshare 未安装，将使用模拟数据")

# --- 中文字体配置 ---
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
np.random.seed(42)
random.seed(42)

# --- 模拟数据生成器 ---
def generate_index_data(days=1500, start_price=3500):
    """生成逼真的沪深300走势"""
    dates = pd.date_range("2021-01-04", periods=days, freq="B")
    # 模拟真实走势
    returns = np.zeros(days)
    # 2021:震荡 - 2022:熊市 - 2023:反弹 - 2024:震荡 - 2025:横盘
    for i in range(days):
        if i < 250:   trend = -0.0001
        elif i < 500: trend = -0.0008
        elif i < 750: trend = -0.0003
        elif i < 1000:trend = 0.0003
        elif i < 1250:trend = 0.0001
        else:         trend = -0.00005
        returns[i] = trend + 0.012 * np.random.randn()
    # 偶尔的极端行情
    shock_idx = np.random.choice(days, size=40, replace=False)
    for idx in shock_idx:
        magnitude = np.random.uniform(0.015, 0.04)
        returns[idx] = magnitude if np.random.random() > 0.6 else -magnitude
    close = start_price * np.exp(np.cumsum(returns))
    open_p = close * (1 + np.random.randn(days) * 0.003)
    high = np.maximum(open_p, close) * (1 + np.abs(np.random.randn(days) * 0.005))
    low = np.minimum(open_p, close) * (1 - np.abs(np.random.randn(days) * 0.005))
    volume = np.abs(np.random.randn(days) * 1e8 + 2e8).astype(int)
    amount = volume * close * 0.01
    return pd.DataFrame({
        "date": dates, "open": open_p.round(2), "close": close.round(2),
        "high": high.round(2), "low": low.round(2),
        "volume": volume, "amount": amount
    })

def generate_stock_data(base_close, days=1200, beta=1.0, vol=0.025):
    """生成个股数据，与指数有 beta 相关性"""
    dates = base_close.index[-days:]
    index_ret = base_close.pct_change().dropna().iloc[-(days-1):].values
    stock_ret = beta * index_ret + np.random.normal(0, vol, len(index_ret))
    stock_close = 100 * np.exp(np.cumsum(stock_ret))
    return pd.Series(stock_close, index=dates[-len(stock_close):], name="close")

# --- 网络容错 ---
def fetch_with_retry(func, name, max_retries=3, base_delay=3):
    """带重试的数据获取"""
    for attempt in range(1, max_retries + 1):
        try:
            print(f"  正在获取 {name} ...", end=" ")
            result = func()
            print("OK")
            return result
        except Exception as e:
            delay = base_delay * (2 ** (attempt - 1))
            if attempt < max_retries:
                print(f"失败({type(e).__name__})，{delay}s 后重试...")
                time.sleep(delay)
            else:
                print(f"失败({type(e).__name__})")
                raise

print(f"pandas {pd.__version__}  |  numpy {np.__version__}")
print(f"akshare: {"可用" if HAS_AKSHARE else "不可用，使用模拟数据"}")


pandas 版本: 3.0.3
numpy 版本: 2.4.6


## 2. 拉取沪深300历史数据

沪深300（CSI 300）是A股最具代表性的宽基指数，包含沪市和深市市值最大的300只股票。

In [3]:
# 获取沪深300指数日线数据（自动降级：在线 → 离线模拟）
print("拉取沪深300指数数据：")
DATA_SOURCE = "未知"

if HAS_AKSHARE:
    try:
        df_index = fetch_with_retry(
            lambda: ak.stock_zh_index_daily_em(symbol="sh000300"),
            name="沪深300日线"
        )
        DATA_SOURCE = "akshare 在线数据"
    except Exception as e:
        print(f"  akashare 不可用，使用本地模拟数据")
        df_index = generate_index_data()
        DATA_SOURCE = "本地模拟(沪深300历史波动特征)"
else:
    df_index = generate_index_data()
    DATA_SOURCE = "本地模拟(沪深300历史波动特征)"

print(f"数据来源: {DATA_SOURCE}")
print(f"数据形状: {df_index.shape}")
print(f"时间范围: {df_index.date.min()} ~ {df_index.date.max()}")
df_index.head(10)


拉取沪深300指数数据：
  正在获取 沪深300日线 ... ✗ 第1次失败 (ConnectionError)，2s 后重试...
  正在获取 沪深300日线 ... ✗ 第2次失败 (ConnectionError)，4s 后重试...
  正在获取 沪深300日线 ... ✗ 第3次失败 (ConnectionError)，8s 后重试...
  正在获取 沪深300日线 ... ✗ 第4次失败 (ConnectionError)，16s 后重试...
  正在获取 沪深300日线 ... ✗ 全部5次重试均失败: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [3]:
# 数据清洗：统一列名，排序
df_index.columns = ['date', 'open', 'close', 'high', 'low', 'volume', 'amount']
df_index['date'] = pd.to_datetime(df_index['date'])
df_index = df_index.sort_values('date').reset_index(drop=True)

# 检查缺失值
print("缺失值统计：")
print(df_index.isnull().sum())

# 基础统计
df_index.describe()

NameError: name 'df_index' is not defined

## 3. 可视化：K线图 + 移动平均线

无需 mplfinance，纯 matplotlib 画 K 线，理解底层原理。

In [ ]:
# 只看最近1年数据（约252个交易日）
df_recent = df_index.tail(252).copy()

# 计算均线
df_recent['MA5'] = df_recent['close'].rolling(window=5).mean()
df_recent['MA20'] = df_recent['close'].rolling(window=20).mean()
df_recent['MA60'] = df_recent['close'].rolling(window=60).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})

# --- 上图：K线 + 均线 ---
ax = axes[0]
colors = ['#ff4d4d' if df_recent['close'].iloc[i] >= df_recent['open'].iloc[i] else '#4dff4d'
          for i in range(len(df_recent))]

# 画K线实体
for i in range(len(df_recent)):
    row = df_recent.iloc[i]
    open_p, close_p, high_p, low_p = row['open'], row['close'], row['high'], row['low']
    body_bottom = min(open_p, close_p)
    body_height = abs(close_p - open_p)
    
    # 上影线
    ax.plot([i, i], [body_bottom + body_height, high_p], color='black', linewidth=0.5)
    # 下影线
    ax.plot([i, i], [low_p, body_bottom], color='black', linewidth=0.5)
    # 实体
    ax.bar(i, body_height, bottom=body_bottom, color=colors[i], width=0.8, alpha=0.8)

# 画均线
ax.plot(range(len(df_recent)), df_recent['MA5'], label='MA5', color='#2196F3', linewidth=1, alpha=0.8)
ax.plot(range(len(df_recent)), df_recent['MA20'], label='MA20', color='#FF9800', linewidth=1.2, alpha=0.8)
ax.plot(range(len(df_recent)), df_recent['MA60'], label='MA60', color='#9C27B0', linewidth=1.2, alpha=0.8)

ax.set_title('沪深300指数 — K线图与移动平均线', fontsize=14, fontweight='bold')
ax.set_ylabel('指数点位', fontsize=11)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

# --- 下图：成交量 ---
ax2 = axes[1]
vol_colors = ['#ff4d4d' if df_recent['close'].iloc[i] >= df_recent['open'].iloc[i] else '#4dff4d'
              for i in range(len(df_recent))]
ax2.bar(range(len(df_recent)), df_recent['volume'] / 10000, color=vol_colors, alpha=0.6, width=0.8)
ax2.set_ylabel('成交量(万手)', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察要点：")
print("1. 短期均线(MA5)在上还是长期均线(MA60)在上 → 判断趋势方向")
print("2. MA5上穿MA20 = 金叉(看涨信号)，下穿 = 死叉(看跌信号)")
print("3. 成交量放大往往伴随趋势加速")

## 4. 收益率分析

In [ ]:
# 计算日收益率
df_index['simple_return'] = df_index['close'].pct_change()           # 简单收益率
df_index['log_return'] = np.log(df_index['close'] / df_index['close'].shift(1))  # 对数收益率

# 去掉第一个 NaN
df_returns = df_index.dropna(subset=['log_return']).copy()

print(f"=== 日收益率统计 ===")
print(f"均值(简单):   {df_returns['simple_return'].mean():.6f}  ({df_returns['simple_return'].mean()*100:.4f}%)")
print(f"均值(对数):   {df_returns['log_return'].mean():.6f}  ({df_returns['log_return'].mean()*100:.4f}%)")
print(f"标准差(日):   {df_returns['log_return'].std():.6f}  ({df_returns['log_return'].std()*100:.4f}%)")
print(f"偏度:        {df_returns['log_return'].skew():.4f}  (负值=左偏，大跌概率>大涨)")
print(f"峰度:        {df_returns['log_return'].kurtosis():.4f}  (>0=肥尾，极端值多于正态分布)")

In [ ]:
# 年化指标
TRADING_DAYS = 252  # A股一年约252个交易日
RISK_FREE_RATE = 0.025  # 假设无风险利率 2.5%（约等于一年期定存）

annual_return = df_returns['log_return'].mean() * TRADING_DAYS       # 年化对数收益率
annual_vol = df_returns['log_return'].std() * np.sqrt(TRADING_DAYS)  # 年化波动率
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_vol          # 夏普比率

print(f"=== 年化指标 ===")
print(f"年化收益率:  {annual_return*100:.2f}%")
print(f"年化波动率:  {annual_vol*100:.2f}%")
print(f"无风险利率:  {RISK_FREE_RATE*100:.1f}%")
print(f"夏普比率:    {sharpe_ratio:.3f}")
print(f"")
print(f"解读：夏普比率 > 1 表示每承担1单位风险获得超过1单位超额收益，算不错。")
print(f"      > 2 表示非常优秀，< 0.5 则需要审视策略质量。")

In [ ]:
# 绘制收益率分布直方图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方图 vs 正态分布
ax = axes[0]
ax.hist(df_returns['log_return'], bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')
# 叠加正态分布
mu = df_returns['log_return'].mean()
sigma = df_returns['log_return'].std()
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
ax.plot(x, 1/(sigma*np.sqrt(2*np.pi)) * np.exp(-(x-mu)**2/(2*sigma**2)), 
        'r-', linewidth=2, label='正态分布')
ax.set_title('日对数收益率分布 vs 正态分布', fontsize=12)
ax.set_xlabel('日收益率')
ax.set_ylabel('概率密度')
ax.legend()
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# 累计收益率曲线
ax2 = axes[1]
cum_return = (1 + df_returns['simple_return']).cumprod() - 1
ax2.plot(df_returns['date'], cum_return * 100, color='steelblue', linewidth=1)
ax2.set_title('沪深300累计收益率', fontsize=12)
ax2.set_xlabel('日期')
ax2.set_ylabel('累计收益率 (%)')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"历史总收益: {cum_return.iloc[-1]*100:.2f}%")

## 5. 最大回撤分析

最大回撤(Max Drawdown)是量化中最重要的风险指标之一——它回答了"最坏情况下你会亏多少"。

In [ ]:
# 计算最大回撤
def calculate_max_drawdown(cum_returns: pd.Series) -> dict:
    """
    计算最大回撤及相关指标
    返回: {mdd, peak_date, trough_date, recovery_date, duration_days}
    """
    running_max = cum_returns.expanding().max()
    drawdown = (cum_returns - running_max) / running_max  # 负值表示回撤
    
    mdd = drawdown.min()
    trough_idx = drawdown.idxmin()
    
    # 找到回撤起点的峰值日期
    peak_idx = cum_returns[:trough_idx].idxmax()
    
    return {
        'max_drawdown': mdd,
        'peak_idx': peak_idx,
        'trough_idx': trough_idx,
        'drawdown_series': drawdown
    }

# 用净值序列计算（初始净值为1）
nav = (1 + df_returns['simple_return']).cumprod()
result = calculate_max_drawdown(nav)

print(f"=== 最大回撤分析 ===")
print(f"最大回撤:  {result['max_drawdown']*100:.2f}%")
print(f"峰值日期:  {df_returns.loc[result['peak_idx'], 'date'].strftime('%Y-%m-%d')}")
print(f"谷底日期:  {df_returns.loc[result['trough_idx'], 'date'].strftime('%Y-%m-%d')}")
print(f"回撤持续:  {(result['trough_idx'] - result['peak_idx']).days} 个自然日")

In [ ]:
# 绘制回撤曲线
fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(df_returns['date'], 0, result['drawdown_series'] * 100, 
                color='#ff4444', alpha=0.3, label='回撤')
ax.plot(df_returns['date'], result['drawdown_series'] * 100, color='#cc0000', linewidth=0.8)
ax.set_title(f'沪深300 回撤曲线 | 最大回撤: {result["max_drawdown"]*100:.2f}%', fontsize=12)
ax.set_ylabel('回撤 (%)')
ax.set_xlabel('日期')
ax.grid(True, alpha=0.3)
ax.legend()

# 标注最大回撤点
ax.annotate(f'最差: {result["max_drawdown"]*100:.1f}%',
            xy=(df_returns.loc[result['trough_idx'], 'date'], result['max_drawdown']*100),
            xytext=(20, -30), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='darkred'),
            fontsize=10, color='darkred', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. 多只股票的相关性矩阵

量化投资的核心思想之一是**分散化**——通过持有多只低相关性的资产，在不降低收益的前提下减少风险。

In [ ]:
# 拉取几只代表性个股的日线数据（失败自动降级为模拟）
stocks = {
    "贵州茅台": {"code": "600519", "beta": 0.95, "vol": 0.022},
    "宁德时代": {"code": "300750", "beta": 1.35, "vol": 0.032},
    "招商银行": {"code": "600036", "beta": 1.05, "vol": 0.020},
    "比亚迪":   {"code": "002594", "beta": 1.25, "vol": 0.030},
    "中国平安": {"code": "601318", "beta": 0.90, "vol": 0.022},
}

close_prices = pd.DataFrame()

for name, cfg in stocks.items():
    code, beta_val, vol_val = cfg["code"], cfg["beta"], cfg["vol"]
    fetched = False
    if HAS_AKSHARE:
        try:
            df_stock = fetch_with_retry(
                lambda c=code: ak.stock_zh_a_hist(symbol=c, period="daily",
                    start_date="20210101", end_date="20250620", adjust="qfq"),
                name=f"{name}({code})"
            )
            df_stock["日期"] = pd.to_datetime(df_stock["日期"])
            df_stock = df_stock.set_index("日期")
            close_prices[name] = df_stock["收盘"]
            fetched = True
            print(f"  {name}: akshare ({len(df_stock)}条)")
        except Exception as e:
            print(f"  {name}: akshare不可用 转为模拟")
    if not fetched:
        base = df_index.set_index("date")["close"]
        close_prices[name] = generate_stock_data(base, beta=beta_val, vol=vol_val)

print(f"\n成功: {len(close_prices.columns)}/{len(stocks)} 只股票, 形状: {close_prices.shape}")
close_prices.head()


In [ ]:
# 计算日收益率（对数）
returns = np.log(close_prices / close_prices.shift(1)).dropna()

# 相关系数矩阵
corr_matrix = returns.corr()

# 可视化
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(stocks)))
ax.set_yticks(range(len(stocks)))
ax.set_xticklabels(stocks.keys(), fontsize=10, rotation=45, ha='right')
ax.set_yticklabels(stocks.keys(), fontsize=10)

# 在每个格子标数值
for i in range(len(stocks)):
    for j in range(len(stocks)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.3f}',
                ha='center', va='center', fontsize=9,
                color='white' if abs(corr_matrix.iloc[i, j]) > 0.4 else 'black')

ax.set_title('个股日收益率相关性矩阵', fontsize=13, fontweight='bold')
plt.colorbar(im, shrink=0.8)
plt.tight_layout()
plt.show()

print("解读：")
print("- 相关性越接近1，同涨同跌越明显 → 分散化效果越差")
print("- 相关性接近0或负值 → 组合在一起可以降低整体波动")
print("- 现实是A股个股间相关性普遍偏高（系统性风险主导）")

## 7. 双均线策略 — 从概念到模拟

这是最经典的量化策略，也是理解策略回测逻辑的最佳起点。

### 策略逻辑
```
IF MA_short 上穿 MA_long（金叉）:
    买入（或空仓→满仓）
ELIF MA_short 下穿 MA_long（死叉）:
    卖出（或满仓→空仓）
ELSE:
    持仓不变
```

### 为什么"简单"的策略也要计算机辅助？
- 肉眼可以看出一两个金叉死叉，但当你有100只股票、10年数据时就看不完了
- 人的情绪会导致执行不纪律（该卖的时候舍不得卖）
- 计算机可以精确统计胜率、盈亏比、最大回撤——用数据而非感觉做决策

In [ ]:
# 在沪深300上模拟双均线策略
SHORT_WINDOW = 5    # 短期均线窗口
LONG_WINDOW = 20    # 长期均线窗口

df_signal = df_index[['date', 'close']].copy()
df_signal['ma_short'] = df_signal['close'].rolling(window=SHORT_WINDOW).mean()
df_signal['ma_long'] = df_signal['close'].rolling(window=LONG_WINDOW).mean()

# 生成信号：1=持有(做多), 0=空仓
df_signal['signal'] = 0
# MA_short > MA_long 时持有
df_signal.loc[df_signal['ma_short'] > df_signal['ma_long'], 'signal'] = 1
# 生成仓位变动（交易点）
df_signal['position_change'] = df_signal['signal'].diff()

# 计算策略的日收益率（当信号=1时吃指数涨跌，信号=0时空仓不赚不亏）
df_signal['benchmark_return'] = df_signal['close'].pct_change()  # 基准买入持有收益
df_signal['strategy_return'] = df_signal['signal'].shift(1) * df_signal['benchmark_return']

# 去掉NaN（均线计算期）
df_ma = df_signal.dropna().copy()

# 累计收益
df_ma['benchmark_nav'] = (1 + df_ma['benchmark_return']).cumprod()
df_ma['strategy_nav'] = (1 + df_ma['strategy_return']).cumprod()

print(f"=== 双均线策略 ({SHORT_WINDOW}/{LONG_WINDOW}) 初步结果 ===")
print(f"信号天数: {len(df_ma)}")
print(f"持仓天数: {df_ma['signal'].sum()} ({df_ma['signal'].mean()*100:.1f}%)")
print(f"交易次数: {df_ma[df_ma['position_change'] != 0].shape[0]}")
print(f"基准累计: {df_ma['benchmark_nav'].iloc[-1]:.4f}")
print(f"策略累计: {df_ma['strategy_nav'].iloc[-1]:.4f}")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(2, 1, figsize=(16, 9))

# --- 上图：价格 + 均线 + 交易信号 ---
ax = axes[0]
ax.plot(df_ma['date'], df_ma['close'], color='black', linewidth=0.8, alpha=0.5, label='收盘价')
ax.plot(df_ma['date'], df_ma['ma_short'], color='#2196F3', linewidth=1.2, label=f'MA{SHORT_WINDOW}')
ax.plot(df_ma['date'], df_ma['ma_long'], color='#FF9800', linewidth=1.5, label=f'MA{LONG_WINDOW}')

# 标注买卖点
buy_signals = df_ma[df_ma['position_change'] == 1]
sell_signals = df_ma[df_ma['position_change'] == -1]
ax.scatter(buy_signals['date'], buy_signals['close'], 
           color='red', marker='^', s=80, zorder=5, label=f'买入({len(buy_signals)}次)', alpha=0.8)
ax.scatter(sell_signals['date'], sell_signals['close'], 
           color='green', marker='v', s=80, zorder=5, label=f'卖出({len(sell_signals)}次)', alpha=0.8)

ax.set_title(f'沪深300 双均线策略 ({SHORT_WINDOW}日/{LONG_WINDOW}日)', fontsize=13, fontweight='bold')
ax.set_ylabel('指数点位')
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

# --- 下图：净值对比 ---
ax2 = axes[1]
ax2.plot(df_ma['date'], df_ma['benchmark_nav'], color='gray', linewidth=1.5, 
         alpha=0.7, label='买入持有 (Benchmark)')
ax2.plot(df_ma['date'], df_ma['strategy_nav'], color='steelblue', linewidth=1.8, 
         label=f'双均线策略 (MA{SHORT_WINDOW}/MA{LONG_WINDOW})')
ax2.set_title('净值曲线对比', fontsize=12)
ax2.set_xlabel('日期')
ax2.set_ylabel('净值 (初始=1)')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=1, color='black', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# 策略绩效对比
def calc_strategy_stats(returns: pd.Series, name: str, rf: float = 0.025):
    """计算策略的关键绩效指标"""
    nav = (1 + returns).cumprod()
    annual_ret = returns.mean() * 252
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = (annual_ret - rf) / annual_vol if annual_vol > 0 else 0
    
    running_max = nav.expanding().max()
    mdd = ((nav - running_max) / running_max).min()
    
    win_rate = (returns > 0).sum() / len(returns)
    avg_win = returns[returns > 0].mean() if len(returns[returns > 0]) > 0 else 0
    avg_loss = returns[returns < 0].mean() if len(returns[returns < 0]) > 0 else 0
    
    print(f"--- {name} ---")
    print(f"  年化收益率: {annual_ret*100:.2f}%")
    print(f"  年化波动率: {annual_vol*100:.2f}%")
    print(f"  夏普比率:   {sharpe:.3f}")
    print(f"  最大回撤:   {mdd*100:.2f}%")
    print(f"  胜率:       {win_rate*100:.1f}%")
    print(f"  平均盈利:   {avg_win*100:.3f}%")
    print(f"  平均亏损:   {avg_loss*100:.3f}%")
    print(f"  盈亏比:     {abs(avg_win/avg_loss):.2f}" if avg_loss != 0 else "  盈亏比: N/A")
    return {'annual_return': annual_ret, 'sharpe': sharpe, 'max_drawdown': mdd, 'win_rate': win_rate}

print("=" * 50)
bench_stats = calc_strategy_stats(df_ma['benchmark_return'], '买入持有(沪深300)')
print()
strategy_stats = calc_strategy_stats(df_ma['strategy_return'], f'双均线策略(MA{SHORT_WINDOW}/{MA{LONG_WINDOW})')

print()
print("=== 策略 vs 基准 ===")
print(f"超额收益: {(strategy_stats['annual_return'] - bench_stats['annual_return'])*100:.2f}%")
print(f"回撤改善: {(bench_stats['max_drawdown'] - strategy_stats['max_drawdown'])*100:.2f}%")

## 8. 总结 & 下一步

通过这个 Notebook，你已经完成了量化投资的第一步：

| 能力 | 掌握情况 |
|------|----------|
| 用 Python 拉取真实金融数据 | ✅ |
| 计算收益率、波动率、夏普比率 | ✅ |
| 理解并计算最大回撤 | ✅ |
| 分析多资产相关性 | ✅ |
| 模拟一个完整的交易策略并评估 | ✅ |

### 关键洞察
1. **趋势跟踪策略在趋势市中表现好，震荡市中反复止损** — 这是双均线的天生缺陷
2. **没有免费的午餐** — 任何策略都有适应的市场环境和不适应的环境
3. **回测 ≠ 实盘** — 回测没有考虑滑点、冲击成本、流动性，实盘会更差

### 下一步
- 尝试修改 SHORT_WINDOW 和 LONG_WINDOW 参数，观察结果变化（这就是参数优化的雏形）
- 阅读《Quantitative Trading》前三章
- 思考：如果加上止损逻辑，结果会变好吗？
- 接下来我们将进入 **阶段1：搭建多市场数据管道**